# 08 — Export Docs

Collects every meaningful output produced by notebooks 01–05 and writes them into
`model_outputs/<batch>/docs/` without modifying any source notebook.

| Section | Source | What is saved |
|---------|--------|---------------|
| nb01 | `data/aggregated/` + `datasets/` | Pair counts, provenance, verification |
| nb02 | `data/dataset/manifest.csv` | Split tables, box counts, structural validation, spot-check grid |
| nb03 | `data/dataset/` labels (live) | Class balance, box-area histogram, dimension scatter, health summary |
| nb04 | `runs/detect/campus_yolo11s/` | results.csv, training curves, batch images, sanity predictions |
| nb05 | `runs/detect/val/` + nb05 JSON | Test metrics, confusion matrices, PR curves, qualitative grid |

In [ ]:
# Cell 2 — Install dependencies
%pip install -q pandas pillow matplotlib seaborn pyyaml
print('nb08 dependencies ready')

In [ ]:
# Cell 3 — Imports
%matplotlib inline

import base64
import io
import json
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import seaborn as sns
import yaml
from PIL import Image

In [ ]:
# Cell 4 — Config
# Set RUN_NAME to a specific folder, or leave None to auto-select the latest batch.
RUN_NAME = None   # e.g. '04_training_batch_1_48am_24_04_2026'

NB_DIR            = Path('.')                               # this notebook lives in notebooks/
DATA_DST          = Path('../data/dataset').resolve()
DATA_AGG          = Path('../data/aggregated').resolve()
DATASETS_SRC      = Path('../datasets').resolve()
MODEL_OUTPUTS_DIR = Path('../model_outputs').resolve()

CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
SPLITS  = ['train', 'val', 'test']

if RUN_NAME is None:
    _runs = sorted([
        d for d in MODEL_OUTPUTS_DIR.iterdir()
        if d.is_dir() and re.match(r'^\d+_training_batch_', d.name)
    ])
    assert _runs, f'No training runs found in {MODEL_OUTPUTS_DIR}'
    RUN_DIR  = _runs[-1]
    RUN_NAME = RUN_DIR.name
else:
    RUN_DIR = MODEL_OUTPUTS_DIR / RUN_NAME

TRAIN_RUNS_DIR = RUN_DIR / 'runs' / 'detect' / 'campus_yolo11s'
VAL_RUNS_DIR   = RUN_DIR / 'runs' / 'detect' / 'val'
DOCS           = RUN_DIR / 'docs'

assert TRAIN_RUNS_DIR.exists(), f'Training run not found: {TRAIN_RUNS_DIR}'
print('Batch :', RUN_NAME)
print('Docs  :', DOCS)
print('Val   :', VAL_RUNS_DIR, '(exists)' if VAL_RUNS_DIR.exists() else '(not yet run)')

## Setup — helpers and folder creation

In [ ]:
# Cell 6 — Helper functions

def mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def load_nb(fname: str) -> list[dict]:
    return json.loads((NB_DIR / fname).read_text(encoding='utf-8'))['cells']

def find_cell(cells: list[dict], cell_id: str) -> dict | None:
    return next((c for c in cells if c.get('id') == cell_id), None)

def extract_png(cells_or_cell, out_path: Path, *, index: int | None = None) -> bool:
    """Save the first image/png output to out_path.
    Pass a cell dict directly, or a cell list + index for ID-less notebooks."""
    cell = cells_or_cell[index] if index is not None else cells_or_cell
    if cell is None:
        print(f'  [NOTE] {out_path.name}: cell not found')
        return False
    for o in cell.get('outputs', []):
        b64 = o.get('data', {}).get('image/png')
        if b64:
            if isinstance(b64, list): b64 = ''.join(b64)
            out_path.write_bytes(base64.b64decode(b64))
            print(f'  [png]  {out_path.name}')
            return True
    print(f'  [NOTE] {out_path.name}: no saved output — re-run source notebook first')
    return False

def cp(src: Path, dst: Path) -> None:
    if src.exists():
        shutil.copy2(src, dst)
        print(f'  [copy] {dst.name}')
    else:
        print(f'  [WARN] {dst.name}: source not found ({src.name})')

def save_csv(df: pd.DataFrame, path: Path, note: str = '') -> None:
    df.to_csv(path, index=True)
    print(f'  [csv]  {path.name}' + (f'  — {note}' if note else ''))

def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    print(f'  [json] {path.name}')


# Create top-level docs dir (subdirs created per section)
mkdir(DOCS)
print('Docs folder ready:', DOCS)

## NB01 — Data Collection

In [ ]:
# Cell 8 — NB01 outputs
print('-- nb01: Data Collection --')
out = mkdir(DOCS / 'nb01_data_collection')

# available_pairs: count image files per class in the raw datasets folder
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
avail = {}
for c in CLASSES:
    src = DATASETS_SRC / c
    avail[c] = sum(1 for p in src.rglob('*') if p.suffix.lower() in IMG_EXTS) if src.exists() else 0

df_pairs = pd.DataFrame(avail.items(), columns=['class', 'available_pairs']).set_index('class')
save_csv(df_pairs, out / 'available_pairs.csv', 'raw (img, label) pairs found before 200-sample')

# source_split_breakdown: read provenance from info.json files written by nb01
rows = []
for c in CLASSES:
    info_path = DATA_AGG / c / 'info.json'
    if info_path.exists():
        info = json.loads(info_path.read_text())
        for item in info.get('items', []):
            rows.append({'class': c,
                         'source_dataset': item['source_dataset'],
                         'source_split':   item['source_split']})

if rows:
    breakdown = (pd.DataFrame(rows)
                   .groupby(['class', 'source_dataset', 'source_split'])
                   .size()
                   .unstack(fill_value=0))
    save_csv(breakdown, out / 'source_split_breakdown.csv',
             'pair counts by class x source_dataset x source_split')

# verification: count aggregated outputs
check = []
for c in CLASSES:
    n_imgs = len(list((DATA_AGG / c / 'images').glob('*.jpg'))) if (DATA_AGG / c / 'images').exists() else 0
    n_lbls = len(list((DATA_AGG / c / 'labels').glob('*.txt'))) if (DATA_AGG / c / 'labels').exists() else 0
    info_ok = (DATA_AGG / c / 'info.json').exists()
    check.append({'class': c, 'images': n_imgs, 'labels': n_lbls, 'info_json': info_ok})
save_csv(pd.DataFrame(check).set_index('class'), out / 'verification.csv',
         '200-per-class aggregation verification')

save_json({
    'notebook': '01_data_collection.ipynb',
    'description': 'Aggregate 200 (image, label) pairs per class from raw Roboflow exports into data/aggregated/. Labels keep class_id=0.',
    'outputs': {
        'available_pairs.csv':       'Available (img, label) pairs found per class before sampling to 200.',
        'source_split_breakdown.csv':'Pair counts broken down by class, source dataset, and source split.',
        'verification.csv':          'Post-run verification: 200 images, 200 labels, info.json per class.',
    },
    'key_facts': {'target_per_class': 200, 'total_images': 800, 'seed': 42}
}, out / 'metadata.json')

## NB02 — Data Annotation

In [ ]:
# Cell 10 — NB02 outputs
print('-- nb02: Data Annotation --')
out = mkdir(DOCS / 'nb02_data_annotation')
cells_nb02 = load_nb('02_data_annotation.ipynb')

# Load manifest.csv — single source of truth for split/class/boxes
manifest = pd.read_csv(DATA_DST / 'manifest.csv')

# stratified_split.csv — images per class per split
split_tbl = (manifest.groupby(['split', 'class'])
                     .size()
                     .unstack(fill_value=0)
                     .reindex(SPLITS))
save_csv(split_tbl, out / 'stratified_split.csv', '70/20/10 stratified image counts per class')

# boxes_per_split_class.csv — images + boxes per (split, class)
boxes_tbl = (manifest.groupby(['split', 'class'])
                     .agg(images=('filename', 'count'), boxes=('boxes', 'sum')))
save_csv(boxes_tbl, out / 'boxes_per_split_class.csv', 'image and box counts per (split, class)')

# structural_validation.csv — walk every label file
checked = {s: {'images': 0, 'labels': 0, 'rows': 0} for s in SPLITS}
issues = []
for split in SPLITS:
    img_dir = DATA_DST / 'images' / split
    lbl_dir = DATA_DST / 'labels' / split
    for img in img_dir.glob('*.jpg'):
        checked[split]['images'] += 1
        lbl = lbl_dir / (img.stem + '.txt')
        if not lbl.exists():
            issues.append((split, img.name, 'missing_label'))
            continue
        checked[split]['labels'] += 1
        for ln, line in enumerate(lbl.read_text().strip().splitlines(), 1):
            checked[split]['rows'] += 1
            parts = line.split()
            if len(parts) != 5:
                issues.append((split, img.name, f'bad_row_{ln}'))

cov = pd.DataFrame(checked).T.rename_axis('split')
cov.loc['total'] = cov.sum()
verdict = 'PASS' if not issues else f'FAIL ({len(issues)} issues)'
save_csv(cov, out / 'structural_validation.csv', f'label coverage — {verdict}')

# spot_check_grid.png — extract from saved nb02 cell output (id: d5dc0aaa)
extract_png(find_cell(cells_nb02, 'd5dc0aaa'), out / 'spot_check_grid.png')

# data.yaml
cp(DATA_DST / 'data.yaml', out / 'data.yaml')

total_imgs  = int(cov.loc['total', 'images'])
total_boxes = int(manifest['boxes'].sum())
save_json({
    'notebook': '02_data_annotation.ipynb',
    'description': 'Merge per-class folders into one stratified 70/20/10 split. Remap class_ids. Write manifest.csv and data.yaml.',
    'outputs': {
        'stratified_split.csv':      'Image counts per class per split.',
        'boxes_per_split_class.csv': 'Image and bounding-box counts per (split, class).',
        'structural_validation.csv': f'Label coverage table. Result: {verdict}.',
        'spot_check_grid.png':       '3x4 grid: 3 random train images per class with bounding boxes overlaid.',
        'data.yaml':                 'Ultralytics data config: paths, split names, class index mapping.',
    },
    'key_facts': {
        'split_ratios':      {'train': 0.70, 'val': 0.20, 'test': 0.10},
        'total_images':      total_imgs,
        'total_boxes':       total_boxes,
        'class_map':         {c: i for i, c in enumerate(CLASSES)},
        'validation_result': verdict,
    }
}, out / 'metadata.json')

## NB03 — Dataset Health Check

In [ ]:
# Cell 12 — NB03 outputs (tables computed live from label files)
print('-- nb03: Dataset Health --')
out = mkdir(DOCS / 'nb03_dataset_health')
cells_nb03 = load_nb('03_data_preprocessing_split.ipynb')
TINY_BOX_AREA = 1e-3

# Build box-level DataFrame (same logic as nb03 cell 6)
rows = []
for split in SPLITS:
    img_dir = DATA_DST / 'images' / split
    lbl_dir = DATA_DST / 'labels' / split
    for img in img_dir.glob('*.jpg'):
        with Image.open(img) as im:
            W, H = im.size
        lbl = lbl_dir / (img.stem + '.txt')
        boxes_found = 0
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5: continue
                cid, xc, yc, w, h = map(float, parts)
                rows.append({'split': split, 'file': img.name, 'W': W, 'H': H,
                             'class': CLASSES[int(cid)], 'bx_w': w, 'bx_h': h, 'bx_area': w * h})
                boxes_found += 1
        if boxes_found == 0:
            rows.append({'split': split, 'file': img.name, 'W': W, 'H': H,
                         'class': None, 'bx_w': None, 'bx_h': None, 'bx_area': None})

df = pd.DataFrame(rows)
print(f'  Loaded {len(df)} rows ({df["bx_area"].notna().sum()} boxes, '
      f'{df["bx_area"].isna().sum()} empty images)')

# raw_data_head.csv
save_csv(df.head(), out / 'raw_data_head.csv', 'first 5 rows of box-level DataFrame')

# class_distribution.csv
per_img = df.dropna(subset=['class']).drop_duplicates(subset=['split', 'file', 'class'])
dist = (per_img.groupby(['split', 'class']).size()
               .unstack(fill_value=0)
               .reindex(SPLITS))
save_csv(dist, out / 'class_distribution.csv', 'images per class per split')

# image_dim_stats.csv
dims = df.drop_duplicates(subset=['split', 'file'])[['split', 'W', 'H']]
save_csv(dims[['W', 'H']].describe(), out / 'image_dim_stats.csv',
         'W/H descriptive stats across all unique images')

# dataset_health_summary.csv
by_split = df.groupby('split')
summary = pd.DataFrame({
    'images':       by_split['file'].nunique(),
    'empty_labels': by_split.apply(lambda g: g[g['class'].isna()]['file'].nunique()),
    'boxes':        by_split['bx_area'].count(),
    'tiny_boxes':   by_split.apply(lambda g: int((g['bx_area'] < TINY_BOX_AREA).sum())),
}).reindex(SPLITS)
save_csv(summary, out / 'dataset_health_summary.csv',
         'images / empty labels / boxes / tiny boxes per split')

# leakage check
files = df.drop_duplicates(subset=['split', 'file'])[['split', 'file']]
leaked = files.groupby('file')['split'].nunique()
n_leaked = int((leaked > 1).sum())

total_boxes = int(df['bx_area'].notna().sum())
tiny_boxes  = int((df['bx_area'] < TINY_BOX_AREA).sum())

save_json({
    'notebook': '03_data_preprocessing_split.ipynb',
    'description': 'Diagnostic health check of data/dataset/. No files modified.',
    'outputs': {
        'raw_data_head.csv':           'First 5 rows of the box-level DataFrame.',
        'class_distribution.csv':      'Images per class per split.',
        'class_distribution_bar.png':  'Bar chart of class distribution (extracted from nb03 output).',
        'box_area_histogram.png':      'Density histogram of normalised box areas per class (extracted from nb03 output).',
        'image_dimensions_scatter.png':'W vs H scatter coloured by split (extracted from nb03 output).',
        'image_dim_stats.csv':         'Descriptive statistics for image width and height.',
        'dataset_health_summary.csv':  'Per-split: images, empty labels, boxes, tiny boxes.',
    },
    'health_checklist': {
        'every_class_in_every_split':    bool((dist > 0).all().all()),
        'tiny_boxes_under_5pct':         bool(tiny_boxes / max(total_boxes, 1) <= 0.05),
        'no_cross_split_leakage':        n_leaked == 0,
        'tiny_box_count':                tiny_boxes,
        'cross_split_leaked_files':      n_leaked,
    }
}, out / 'metadata.json')

In [ ]:
# Cell 13 — NB03 inline figures
# nb03 cells have no IDs — use positional index (7, 9, 11)
extract_png(cells_nb03, out / 'class_distribution_bar.png',  index=7)
extract_png(cells_nb03, out / 'box_area_histogram.png',       index=9)
extract_png(cells_nb03, out / 'image_dimensions_scatter.png', index=11)

## NB04 — Model Training

In [ ]:
# Cell 15 — NB04 artifacts
print('-- nb04: Model Training --')
out = mkdir(DOCS / 'nb04_model_training')
cells_nb04 = load_nb('04_model_training.ipynb')

# Copy static artifacts
cp(TRAIN_RUNS_DIR / 'args.yaml',   out / 'training_config.yaml')
cp(TRAIN_RUNS_DIR / 'results.csv', out / 'training_results.csv')
cp(TRAIN_RUNS_DIR / 'labels.jpg',  out / 'label_distribution.jpg')

# Inline figures (IDs present in nb04)
extract_png(find_cell(cells_nb04, 'bc4a8371'), out / 'training_curves.png')
extract_png(find_cell(cells_nb04, '15f8a39d'), out / 'sanity_predictions_grid.png')

# Sanity prediction images
sanity_out = mkdir(out / 'sanity')
for img in sorted((TRAIN_RUNS_DIR / 'sanity').glob('*.jpg')):
    cp(img, sanity_out / img.name)

# Training batch mosaics
tb_out = mkdir(out / 'train_batches')
for img in sorted(TRAIN_RUNS_DIR.glob('train_batch*.jpg')):
    cp(img, tb_out / img.name)

# Validation batch images
vb_out = mkdir(out / 'val_batches')
for img in sorted(TRAIN_RUNS_DIR.glob('val_batch*.jpg')):
    cp(img, vb_out / img.name)

# Compute training_summary.json from results.csv
df_res = pd.read_csv(TRAIN_RUNS_DIR / 'results.csv')
df_res.columns = [c.strip() for c in df_res.columns]
best_row  = df_res.nlargest(1, 'metrics/mAP50(B)').iloc[0]
final_row = df_res.iloc[-1]
summary = {
    'epochs_trained':        int(df_res['epoch'].values[-1]),
    'best_epoch':            int(best_row['epoch']),
    'best_val_mAP50':        round(float(best_row['metrics/mAP50(B)']), 4),
    'best_val_mAP50_95':     round(float(best_row['metrics/mAP50-95(B)']), 4),
    'final_train_box_loss':  round(float(final_row['train/box_loss']), 4),
    'final_train_cls_loss':  round(float(final_row['train/cls_loss']), 4),
    'final_val_box_loss':    round(float(final_row['val/box_loss']), 4),
    'final_val_cls_loss':    round(float(final_row['val/cls_loss']), 4),
}
save_json(summary, out / 'training_summary.json')

save_json({
    'notebook': '04_model_training.ipynb',
    'description': 'Train YOLOv11n on the unified 800-image dataset.',
    'outputs': {
        'training_config.yaml':       'Full Ultralytics training configuration (args.yaml).',
        'training_results.csv':       'Per-epoch training metrics for all epochs.',
        'training_curves.png':        'Train+val loss curves and val mAP curves.',
        'training_summary.json':      'Key stats: best epoch, best mAP, final losses.',
        'label_distribution.jpg':     'YOLO label distribution: class frequencies and box heatmaps.',
        'sanity_predictions_grid.png':'Inline grid of sanity inference predictions.',
        'sanity/':                    'Individual annotated JPEG predictions (sanity inference).',
        'train_batches/':             'Training batch mosaic visualisations.',
        'val_batches/':               'Validation batch ground-truth and prediction images.',
    },
    'training_summary': summary,
}, out / 'metadata.json')

## NB05 — Model Evaluation

In [ ]:
# Cell 17 — NB05 outputs
print('-- nb05: Model Evaluation --')
out = mkdir(DOCS / 'nb05_model_evaluation')
cells_nb05 = load_nb('05_model_evaluation.ipynb')

# ── Parse metrics from saved notebook output (cell ae51a559 = cell 6) ────────
def _parse_val_metrics(cells):
    """Extract overall metrics from nb05 cell 6 stream output."""
    cell = find_cell(cells, 'ae51a559')
    if cell is None:
        return None
    text = ''.join(
        ''.join(o.get('text', []))
        for o in cell.get('outputs', [])
        if o.get('output_type') == 'stream'
    )
    result = {}
    for line in text.splitlines():
        for key, label in [('mAP_at_0.5',      'mAP@0.5'),
                            ('mAP_at_0.5_0.95', 'mAP@0.5:0.95'),
                            ('precision_macro', 'Precision (macro)'),
                            ('recall_macro',    'Recall    (macro)')]:
            if label in line:
                m = re.search(r'([\d.]+)\s*$', line)
                if m:
                    result[key] = round(float(m.group(1)), 4)
    return result if len(result) == 4 else None

def _parse_per_class(cells):
    """Extract per-class metrics from nb05 cell 8 text/plain output."""
    cell = find_cell(cells, '9ba9d4da')
    if cell is None:
        return None
    for o in cell.get('outputs', []):
        text = ''.join(o.get('data', {}).get('text/plain', []))
        if not text:
            continue
        rows = []
        for line in text.strip().splitlines():
            parts = line.split()
            if len(parts) >= 5 and parts[0].isdigit():
                try:
                    _, cls, p, r, ap50, ap = parts[0], parts[1], float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
                    rows.append({'class': cls, 'precision': p, 'recall': r,
                                 'mAP@0.5': ap50, 'mAP@0.5:0.95': ap})
                except (ValueError, IndexError):
                    pass
        if rows:
            return pd.DataFrame(rows).set_index('class')
    return None

def _parse_variant(cells):
    """Extract variant comparison table from nb05 cell 14 text/plain output."""
    cell = find_cell(cells, '98bed36d')
    if cell is None:
        return None
    for o in cell.get('outputs', []):
        text = ''.join(o.get('data', {}).get('text/plain', []))
        if not text:
            continue
        rows = []
        for line in text.strip().splitlines():
            parts = line.split()
            if parts and parts[0].isdigit() and len(parts) >= 3:
                try:
                    variant  = parts[1]
                    params_m = float(parts[2])
                    ap50     = float(parts[3]) if len(parts) > 3 and parts[3] != 'None' else None
                    ap       = float(parts[4]) if len(parts) > 4 and parts[4] != 'None' else None
                    lat      = float(parts[5]) if len(parts) > 5 and parts[5] != 'None' else None
                    rows.append({'variant': variant, 'params_M': params_m,
                                 'mAP@0.5': ap50, 'mAP@0.5:0.95': ap, 'latency_ms': lat})
                except (ValueError, IndexError):
                    pass
        if rows:
            return pd.DataFrame(rows).set_index('variant')
    return None

# ── Pull parsed data ──────────────────────────────────────────────────────────
overall_parsed   = _parse_val_metrics(cells_nb05)
per_class_parsed = _parse_per_class(cells_nb05)
variant_parsed   = _parse_variant(cells_nb05)

if overall_parsed:
    overall_parsed.update({'model': 'yolo11n', 'conf_threshold': 0.25,
                           'iou_threshold': 0.5, 'imgsz': 640})
    save_json(overall_parsed, out / 'overall_metrics.json')
else:
    print('  [NOTE] overall_metrics.json: re-run nb05 to populate')

if per_class_parsed is not None:
    save_csv(per_class_parsed, out / 'per_class_metrics.csv',
             'per-class precision/recall/mAP on the test split')
else:
    print('  [NOTE] per_class_metrics.csv: re-run nb05 to populate')

if variant_parsed is not None:
    save_csv(variant_parsed, out / 'variant_comparison.csv',
             'model variant comparison (yolo11s/yolo11m rows pending)')
else:
    print('  [NOTE] variant_comparison.csv: re-run nb05 to populate')

# ── Inline figures ────────────────────────────────────────────────────────────
extract_png(find_cell(cells_nb05, '5442b1f9'), out / 'confusion_matrix_custom.png')
extract_png(find_cell(cells_nb05, '3708775d'), out / 'pr_curve_display.png')
extract_png(find_cell(cells_nb05, '42711672'), out / 'qualitative_predictions.png')

# ── Curve / matrix PNGs — prefer val run, fall back to training run ───────────
curve_src = VAL_RUNS_DIR if VAL_RUNS_DIR.exists() else TRAIN_RUNS_DIR
print(f'  Curves sourced from: {curve_src.name}')
for fname in ['confusion_matrix.png', 'confusion_matrix_normalized.png',
              'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png']:
    cp(curve_src / fname, out / fname)

test_results = overall_parsed or {}
if per_class_parsed is not None:
    test_results['per_class'] = per_class_parsed.to_dict(orient='index')

save_json({
    'notebook': '05_model_evaluation.ipynb',
    'description': 'Evaluate best checkpoint on the held-out test split.',
    'outputs': {
        'overall_metrics.json':           'Scalar test-set metrics: mAP@0.5, mAP@0.5:0.95, precision, recall.',
        'per_class_metrics.csv':          'Per-class precision/recall/mAP for all 4 classes.',
        'confusion_matrix_custom.png':    'Custom seaborn dual-panel confusion matrix (counts + normalised).',
        'pr_curve_display.png':           'Per-class PR curves displayed inline.',
        'qualitative_predictions.png':    '2x4 grid of test images with predicted bounding boxes.',
        'confusion_matrix.png':           'YOLO-generated confusion matrix.',
        'confusion_matrix_normalized.png':'YOLO-generated column-normalised confusion matrix.',
        'BoxPR_curve.png':                'Per-class precision-recall curves.',
        'BoxP_curve.png':                 'Precision vs confidence threshold per class.',
        'BoxR_curve.png':                 'Recall vs confidence threshold per class.',
        'BoxF1_curve.png':                'F1 vs confidence threshold per class.',
        'variant_comparison.csv':         'Model variant comparison (yolo11s/yolo11m rows pending).',
    },
    'test_results': test_results,
    'curve_source': curve_src.name,
}, out / 'metadata.json')

## Summary

In [ ]:
# Cell 19 — Write README.md and print final summary

# Build README content from live data
m = overall_parsed or {}
pc = per_class_parsed

def _fmt(v, decimals=4):
    return f'{v:.{decimals}f}' if isinstance(v, float) else str(v) if v is not None else 'pending'

pc_rows = ''
if pc is not None:
    for cls, row in pc.iterrows():
        pc_rows += (f'| {cls} | {_fmt(row.get("precision"))} | '
                    f'{_fmt(row.get("recall"))} | {_fmt(row.get("mAP@0.5"))} | '
                    f'{_fmt(row.get("mAP@0.5:0.95"))} |\n')
else:
    pc_rows = '| — | — | — | — | — |\n'

# Dataset split summary from manifest
manifest = pd.read_csv(DATA_DST / 'manifest.csv')
split_summary = manifest.groupby('split').agg(images=('filename','count'), boxes=('boxes','sum'))
total_row = split_summary.sum().rename('total')
split_summary = pd.concat([split_summary, total_row.to_frame().T])
split_tbl_md = '| Split | Images | Boxes |\n|-------|--------|-------|\n'
for idx, row in split_summary.iterrows():
    split_tbl_md += f'| **{idx}** | {int(row["images"])} | {int(row["boxes"])} |\n'

readme = f"""# Batch Docs — {RUN_NAME}

| Folder | Notebook | Contents |
|--------|----------|----------|
| `nb01_data_collection/`  | 01 | Pair counts, provenance, aggregation verification |
| `nb02_data_annotation/`  | 02 | Split tables, box counts, structural validation, spot-check grid |
| `nb03_dataset_health/`   | 03 | Class-balance chart, box-area histogram, dimension scatter, health summary |
| `nb04_model_training/`   | 04 | Training config, results.csv, loss+mAP curves, batch images, sanity predictions |
| `nb05_model_evaluation/` | 05 | Test metrics, confusion matrices, PR curves, qualitative predictions |

---

## Test Results (nb05)

| Metric | Value |
|--------|-------|
| mAP@0.5 | {_fmt(m.get('mAP_at_0.5'))} |
| mAP@0.5:0.95 | {_fmt(m.get('mAP_at_0.5_0.95'))} |
| Precision (macro) | {_fmt(m.get('precision_macro'))} |
| Recall (macro) | {_fmt(m.get('recall_macro'))} |

### Per-Class

| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|-------|-----------|--------|---------|---------------|
{pc_rows}
---

## Dataset

{split_tbl_md}
Classes: `projector=0`, `whiteboard=1`, `fire_extinguisher=2`, `door_sign=3`
"""

(DOCS / 'README.md').write_text(readme, encoding='utf-8')
print('  [md]   README.md')

# Final file tree
print(f'\nDocs saved to: {DOCS}')
print(f'Total files  : {sum(1 for _ in DOCS.rglob("*") if _.is_file())}')
print()
for section in sorted(DOCS.iterdir()):
    if section.is_dir():
        files = sorted(section.rglob('*'))
        print(f'  {section.name}/  ({sum(1 for f in files if f.is_file())} files)')
        for f in files:
            if f.is_file():
                rel = f.relative_to(section)
                size_kb = f.stat().st_size / 1024
                print(f'    {rel}  ({size_kb:.1f} KB)')
    else:
        size_kb = section.stat().st_size / 1024
        print(f'  {section.name}  ({size_kb:.1f} KB)')